# Lab 2: Music Genre Classification - CNNs
Este notebook contiene la solución de las Tareas 1, 2 y 3 del laboratorio:

- **Task 1**: CNN entrenada desde cero sobre los espectrogramas mel, explorando distintas configuraciones (nº de capas convolucionales y de filtros).
- **Task 2**: modelos preentrenados en ImageNet (ResNet18 y VGG16):
  - *embeddings* congelados + regresión logística;
  - fine-tuning completo.
- **Task 3**: research questions Q1, Q2 y Q5, comparando con los modelos del Lab 1 reentrenados sobre **la misma partición 80/20**.

**Requisitos**: el notebook está pensado para **GPU NVIDIA (CUDA)**; en CPU funciona, pero es muy lento.
- PyTorch con CUDA: elige el comando para tu versión de CUDA en pytorch.org, p. ej. `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128`.
- Resto de librerías: `pip install pandas scikit-learn matplotlib seaborn pillow`.

In [ ]:
import os, time, copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torchvision

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

# En Colab: montar Drive y apuntar DATA_DIR a la carpeta "Data"
# from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = 'Data-20261104T120000Z-1-001/Data'
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'  # precisión mixta (float16): más rápido y menos memoria en GPU
torch.backends.cudnn.benchmark = True  # cuDNN elige el algoritmo de convolución más rápido para cada tamaño

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} ({props.total_memory / 1e9:.1f} GB) | PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
else:
    print('AVISO: no se detecta GPU CUDA; el entrenamiento en CPU será muy lento')

## Carga de los espectrogramas
Las imágenes de `images_original` son PNG de 432×288 generadas con matplotlib, con un marco blanco alrededor del espectrograma. Se recorta ese marco (la caja es la misma en todas las imágenes) y se preparan dos versiones:

- **CNN desde cero**: escala de grises a media resolución (109×168). El mapa de color *magma* es monótono en luminancia, así que la escala de grises conserva la intensidad.
- **Modelos preentrenados**: RGB 224×224, el formato con el que se entrenaron en ImageNet.

El dataset tiene 999 canciones: `jazz.00054.wav` está corrupto en GTZAN y no tiene espectrograma.

In [ ]:
CROP = (54, 35, 390, 253)  # (izq, arriba, der, abajo): elimina el marco blanco

img_dir = os.path.join(DATA_DIR, 'images_original')
GENRES = sorted(g for g in os.listdir(img_dir) if os.path.isdir(os.path.join(img_dir, g)))

records = []
for genre in GENRES:
    for fname in sorted(os.listdir(os.path.join(img_dir, genre))):
        if fname.endswith('.png'):
            records.append({
                'path': os.path.join(img_dir, genre, fname),
                'genre': genre,
                'song': fname.replace(genre, genre + '.').replace('.png', '.wav'),  # blues00000.png -> blues.00000.wav
            })
meta = pd.DataFrame(records)

le = LabelEncoder()
y = le.fit_transform(meta['genre'])
genre_names = le.classes_

imgs = [Image.open(p).convert('RGB').crop(CROP) for p in meta['path']]
X_small = np.stack([np.array(im.convert('L').resize((168, 109))) for im in imgs])[:, None]  # (N, 1, 109, 168)
X_224 = np.stack([np.array(im.resize((224, 224))) for im in imgs]).transpose(0, 3, 1, 2)  # (N, 3, 224, 224)
del imgs

print('Canciones:', len(meta))
print('X_small:', X_small.shape, '| X_224:', X_224.shape)
print(meta['genre'].value_counts().sort_index().to_dict())

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 5))
for ax, genre in zip(axes.flat, GENRES):
    i = meta.index[meta['genre'] == genre][0]
    ax.imshow(X_small[i, 0], cmap='magma', aspect='auto')
    ax.set_title(genre)
    ax.axis('off')
plt.suptitle('Un espectrograma por género (entrada de la CNN desde cero)')
plt.tight_layout()
plt.show()

## Partición 80/20
Se barajan los datos y se hace una partición estratificada 80% train / 20% test (200 canciones). **Todos** los modelos del notebook (CNNs, modelos preentrenados y modelos del Lab 1) usan exactamente la misma partición.

Para elegir configuraciones **sin tocar el test**, el 80% de train se divide a su vez en 80/20 (train interno / validación). La validación decide la configuración y la época de *early stopping*.

Como hay un único espectrograma por canción, no hay fuga de datos entre particiones (a diferencia de los fragmentos de 3 s del Lab 1).

In [ ]:
def make_split(seed):
    tr_all, te = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, shuffle=True, random_state=seed)
    tr, va = train_test_split(tr_all, test_size=0.2, stratify=y[tr_all], shuffle=True, random_state=seed)
    return tr_all, tr, va, te

idx_train, idx_tr, idx_val, idx_test = make_split(SEED)
print(f'Train: {len(idx_train)} (train interno {len(idx_tr)} + validación {len(idx_val)}) | Test: {len(idx_test)}')

# Resultados de todos los modelos sobre el test de 200 canciones
results = []
test_preds = {}

## Utilidades de entrenamiento
- Los datos están precargados en memoria como `uint8`. Cada *batch* se envía a la GPU y se normaliza allí.
- **Precisión mixta (AMP)**: en GPU, el *forward* se hace en float16 con `torch.autocast` y `GradScaler`. Es aproximadamente 2× más rápido y usa menos memoria, sin pérdida apreciable de accuracy.
- `train_model`: Adam + *cross-entropy*. Guarda los pesos de la mejor época de validación y para cuando deja de mejorar (*early stopping*).
- **Data augmentation**: desplazamiento circular aleatorio en el eje temporal. El género no cambia si la canción empieza unos segundos antes o después.
  - No se usan *flips*: un flip horizontal invierte el tiempo y uno vertical invierte las frecuencias, y ninguno tiene sentido musical.

In [ ]:
def make_loader(X, idx, batch_size=32, shuffle=False):
    ds = TensorDataset(torch.as_tensor(X[idx]), torch.as_tensor(y[idx]))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=DEVICE.type == 'cuda')

def prep_small(xb):
    return xb.float() / 255.0

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def prep_imagenet(xb):
    return (xb.float() / 255.0 - IMAGENET_MEAN) / IMAGENET_STD

def random_time_shift(xb, max_frac=0.25):
    max_shift = int(xb.shape[-1] * max_frac)
    return torch.roll(xb, shifts=np.random.randint(-max_shift, max_shift + 1), dims=-1)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

@torch.no_grad()
def predict(model, prep, loader):
    model.eval()
    ys, preds = [], []
    for xb, yb in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(prep(xb.to(DEVICE, non_blocking=True)))
        preds.append(logits.argmax(1).cpu())
        ys.append(yb)
    return torch.cat(ys).numpy(), torch.cat(preds).numpy()

def train_model(model, prep, train_loader, val_loader, epochs=40, lr=1e-3, weight_decay=1e-4, patience=8, augment=True):
    model = model.to(DEVICE)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    history = []
    best_acc, best_epoch, best_state, wait = -1, 0, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum, n = 0.0, 0
        for xb, yb in train_loader:
            xb = prep(xb.to(DEVICE, non_blocking=True))
            yb = yb.to(DEVICE, non_blocking=True)
            if augment:
                xb = random_time_shift(xb)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = F.cross_entropy(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            loss_sum += loss.item() * len(yb)
            n += len(yb)

        val_acc = accuracy_score(*predict(model, prep, val_loader))
        history.append({'epoch': epoch, 'train_loss': loss_sum / n, 'val_acc': val_acc})
        if val_acc > best_acc:
            # copia en CPU para no duplicar el modelo en la memoria de la GPU
            best_acc, best_epoch, wait = val_acc, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch, best_acc

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def grid_eval(model, params, X, tr, te):
    # Escalado + búsqueda de hiperparámetros con CV dentro de train, evaluación en test
    grid = GridSearchCV(Pipeline([('scaler', StandardScaler()), ('clf', model)]),
                        {f'clf__{k}': v for k, v in params.items()}, cv=cv, scoring='accuracy', n_jobs=-1)
    grid.fit(X[tr], y[tr])
    y_pred = grid.predict(X[te])
    return grid, y_pred, accuracy_score(y[te], y_pred)

---
# Task 1: CNN desde cero
Arquitectura base: *N* bloques `Conv 3×3 → BatchNorm → ReLU → MaxPool 2×2`. El número de filtros se duplica en cada bloque (p. ej. 16-32-64).

La cabeza es *Global Average Pooling* → Dropout → capa lineal de 10 clases. El GAP evita la capa densa gigante de VGG y mantiene pocos parámetros, que es importante con solo ~640 canciones de entrenamiento.

Se exploran **2, 3, 4 y 5 capas convolucionales** con **16 o 32 filtros** iniciales. La elección se hace por accuracy de **validación**.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_conv=3, base_filters=16, n_classes=10, dropout=0.3):
        super().__init__()
        layers, in_ch = [], 1
        for i in range(n_conv):
            out_ch = base_filters * 2 ** i
            layers += [nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(), nn.MaxPool2d(2)]
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(in_ch, n_classes))

    def forward(self, x):
        return self.head(self.features(x))

train_loader = make_loader(X_small, idx_tr, shuffle=True)
val_loader = make_loader(X_small, idx_val)
test_loader = make_loader(X_small, idx_test)

grid_rows, cnn_runs = [], {}
for n_conv in [2, 3, 4, 5]:
    for base_filters in [16, 32]:
        name = f'CNN {n_conv} capas / {base_filters} filtros'
        set_seed()
        start = time.time()
        model, hist, best_epoch, best_acc = train_model(SimpleCNN(n_conv, base_filters), prep_small, train_loader, val_loader)
        elapsed = time.time() - start
        cnn_runs[name] = (model, hist)
        grid_rows.append({'Modelo': name, 'Capas conv': n_conv, 'Filtros iniciales': base_filters,
                          'Parámetros': count_params(model), 'Mejor época': best_epoch,
                          'Val accuracy': best_acc, 'Tiempo (s)': elapsed})
        print(f'{name}: val acc = {best_acc:.3f} (época {best_epoch}, {elapsed:.0f}s)')

grid_df = pd.DataFrame(grid_rows).sort_values('Val accuracy', ascending=False).reset_index(drop=True)
display(grid_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for base_filters, marker in [(16, 'o'), (32, 's')]:
    sub = grid_df[grid_df['Filtros iniciales'] == base_filters].sort_values('Parámetros')
    axes[0].plot(sub['Parámetros'], sub['Val accuracy'], marker=marker, label=f'{base_filters} filtros iniciales')
    for _, r in sub.iterrows():
        axes[0].annotate(f"{r['Capas conv']}c", (r['Parámetros'], r['Val accuracy']), textcoords='offset points', xytext=(5, 5))
axes[0].set_xscale('log')
axes[0].set_xlabel('Nº de parámetros (log)')
axes[0].set_ylabel('Accuracy de validación')
axes[0].set_title('Profundidad / tamaño vs accuracy')
axes[0].legend()

best_cnn_name = grid_df.loc[0, 'Modelo']
hist = cnn_runs[best_cnn_name][1]
axes[1].plot(hist['epoch'], hist['train_loss'], label='Train loss')
ax2 = axes[1].twinx()
ax2.plot(hist['epoch'], hist['val_acc'], color='tab:orange', label='Val accuracy')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Train loss')
ax2.set_ylabel('Val accuracy')
axes[1].set_title(f'Curvas de entrenamiento: {best_cnn_name}')
plt.tight_layout()
plt.show()

### Evaluación final de la mejor configuración en test
Solo ahora, con la configuración ya elegida por validación, se evalúa una única vez sobre las 200 canciones de test.

In [ ]:
best_cnn = cnn_runs[best_cnn_name][0]
best_row = grid_df.loc[0]
y_true, y_pred = predict(best_cnn, prep_small, test_loader)
test_preds['CNN desde cero'] = y_pred

results.append({'Modelo': f'{best_cnn_name} (desde cero)', 'Familia': 'CNN desde cero',
                'Test accuracy': accuracy_score(y_true, y_pred), 'Parámetros': best_row['Parámetros'],
                'Tiempo entrenamiento (s)': best_row['Tiempo (s)']})
print(f'{best_cnn_name} -> test accuracy = {accuracy_score(y_true, y_pred):.3f}')

---
# Task 2: modelos preentrenados
Se usan redes entrenadas en **ImageNet** (`torchvision`) con dos estrategias:

1. **Embeddings congelados + clasificador clásico**: la red preentrenada se usa solo como extractor de características y no se modifica ningún peso. Sobre los vectores extraídos se entrena una regresión logística, como en el Lab 1.
   - **ResNet18**: vector de 512 dimensiones tras el *Global Average Pooling*.
   - **VGG16**: vector de 4096 dimensiones de la penúltima capa densa.
2. **Fine-tuning completo**:
   - Se sustituye la capa final (1000 clases de ImageNet → 10 géneros).
   - Se reentrena **toda** la red con un *learning rate* bajo para no destruir lo que ya sabe. VGG16 usa uno aún más bajo porque es más grande y más inestable.
   - El early stopping usa la misma validación que la Task 1.

## 2a. Embeddings congelados + regresión logística

In [ ]:
@torch.no_grad()
def extract_features(backbone, X, batch_size=64):
    backbone = backbone.to(DEVICE).eval()
    feats = []
    for i in range(0, len(X), batch_size):
        xb = prep_imagenet(torch.as_tensor(X[i:i + batch_size]).to(DEVICE, non_blocking=True))
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            feats.append(backbone(xb).float().cpu())
    return torch.cat(feats).numpy()

resnet_backbone = torchvision.models.resnet18(weights='IMAGENET1K_V1')
resnet_backbone.fc = nn.Identity()  # salida: embedding de 512

vgg_backbone = torchvision.models.vgg16(weights='IMAGENET1K_V1')
vgg_backbone.classifier = vgg_backbone.classifier[:-1]  # quita la capa de 1000 clases -> embedding de 4096

LOGREG_GRID = {'C': [0.001, 0.01, 0.1, 1]}
embeddings = {}
for name, backbone in [('ResNet18', resnet_backbone), ('VGG16', vgg_backbone)]:
    start = time.time()
    feats = extract_features(backbone, X_224)
    t_extract = time.time() - start
    embeddings[name] = feats

    start = time.time()
    clf, y_pred, acc = grid_eval(LogisticRegression(max_iter=2000), LOGREG_GRID, feats, idx_train, idx_test)
    t_fit = time.time() - start
    test_preds[f'{name} congelado + LogReg'] = y_pred
    results.append({'Modelo': f'{name} congelado + LogReg', 'Familia': 'Preentrenado (congelado)',
                    'Test accuracy': acc, 'Parámetros': count_params(backbone),
                    'Tiempo entrenamiento (s)': t_extract + t_fit})
    print(f'{name}: embedding {feats.shape[1]}-d | CV acc = {clf.best_score_:.3f} con {clf.best_params_} '
          f'| test acc = {acc:.3f} | extracción {t_extract:.1f}s')

del resnet_backbone, vgg_backbone
torch.cuda.empty_cache()

## 2b. Fine-tuning completo

In [ ]:
FT_CONFIG = {
    'ResNet18': {'lr': 1e-4, 'batch_size': 32},
    'VGG16': {'lr': 2e-5, 'batch_size': 16},  # baja a 8 si la GPU se queda sin memoria
}

def build_pretrained(arch):
    if arch == 'ResNet18':
        model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, len(GENRES))  # nueva cabeza para 10 géneros
    else:
        model = torchvision.models.vgg16(weights='IMAGENET1K_V1')
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(GENRES))
    return model

def finetune(arch, tr, va, seed=SEED, epochs=15, patience=4):
    set_seed(seed)
    cfg = FT_CONFIG[arch]
    return train_model(build_pretrained(arch), prep_imagenet,
                       make_loader(X_224, tr, batch_size=cfg['batch_size'], shuffle=True),
                       make_loader(X_224, va, batch_size=cfg['batch_size']),
                       epochs=epochs, lr=cfg['lr'], patience=patience)

ft_runs = {}
for arch in FT_CONFIG:
    start = time.time()
    model, hist, best_epoch, best_acc = finetune(arch, idx_tr, idx_val)
    elapsed = time.time() - start

    y_true, y_pred = predict(model, prep_imagenet, make_loader(X_224, idx_test, batch_size=64))
    name = f'{arch} fine-tuning'
    test_preds[name] = y_pred
    ft_runs[name] = {'hist': hist, 'val_acc': best_acc}
    results.append({'Modelo': f'{arch} fine-tuning completo', 'Familia': 'Preentrenado (fine-tuning)',
                    'Test accuracy': accuracy_score(y_true, y_pred), 'Parámetros': count_params(model),
                    'Tiempo entrenamiento (s)': elapsed})
    print(f'{name}: val acc = {best_acc:.3f} (época {best_epoch}) | test acc = {accuracy_score(y_true, y_pred):.3f} '
          f'| tiempo = {elapsed:.0f}s')

    del model
    torch.cuda.empty_cache()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, run in ft_runs.items():
    ax.plot(run['hist']['epoch'], run['hist']['val_acc'], marker='o', label=name)
hist_cnn = cnn_runs[best_cnn_name][1]
ax.plot(hist_cnn['epoch'], hist_cnn['val_acc'], marker='.', label=f'{best_cnn_name} (desde cero)')
ax.set_xlabel('Época')
ax.set_ylabel('Accuracy de validación')
ax.set_title('Preentrenado vs desde cero')
ax.legend()
plt.show()

---
# Task 3: Research questions
Se responden las preguntas **Q1, Q2 y Q5**. Para comparar de forma justa, primero se reentrenan los modelos del Lab 1 sobre la misma partición:
- mismos algoritmos y rejillas de hiperparámetros que el Lab 1, sobre `features_30_sec.csv`;
- búsqueda de hiperparámetros con CV de 5 *folds* dentro del 80% de train;
- evaluación sobre las mismas 200 canciones de test.

In [ ]:
df30 = pd.read_csv(os.path.join(DATA_DIR, 'features_30_sec.csv')).set_index('filename')
df30 = df30.loc[meta['song']]  # mismo orden que las imágenes (sin jazz.00054)
assert (df30['label'].values == meta['genre'].values).all()
X_desc = df30.drop(columns=['label', 'length']).values

lab1_models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000), {'C': [0.1, 1, 10]}),
    'k-NN': (KNeighborsClassifier(), {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']}),
    'Decision Tree': (DecisionTreeClassifier(random_state=SEED), {'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]}),
    'Random Forest': (RandomForestClassifier(random_state=SEED), {'n_estimators': [50, 100], 'max_depth': [None, 10]}),
    'ANN': (MLPClassifier(max_iter=500, random_state=SEED), {'hidden_layer_sizes': [(50,), (100,)], 'alpha': [0.0001, 0.001]}),
}

lab1_rows = []
for name, (model, params) in lab1_models.items():
    start = time.time()
    grid, y_pred, acc = grid_eval(model, params, X_desc, idx_train, idx_test)
    elapsed = time.time() - start
    lab1_rows.append({'name': name, 'cv_acc': grid.best_score_, 'pred': y_pred})
    results.append({'Modelo': f'{name} (descriptores Lab 1)', 'Familia': 'Descriptores manuales',
                    'Test accuracy': acc, 'Parámetros': np.nan, 'Tiempo entrenamiento (s)': elapsed})
    print(f'{name}: CV acc = {grid.best_score_:.3f} | test acc = {acc:.3f}')

# El "mejor modelo del Lab 1" se elige por CV, no por test
best_lab1 = max(lab1_rows, key=lambda r: r['cv_acc'])
best_lab1_name = f"{best_lab1['name']} (Lab 1)"
test_preds[best_lab1_name] = best_lab1['pred']
print('Mejor modelo de descriptores (por CV):', best_lab1['name'])

## Q1. ¿Las features aprendidas superan a los descriptores manuales?
Todos los modelos evaluados sobre el **mismo test de 200 canciones**. Una canción equivale a 0.5 puntos de accuracy.

In [ ]:
summary = pd.DataFrame(results).sort_values('Test accuracy', ascending=False).reset_index(drop=True)
display(summary.style.format({'Test accuracy': '{:.3f}', 'Parámetros': '{:,.0f}', 'Tiempo entrenamiento (s)': '{:.1f}'}))

palette = dict(zip(summary['Familia'].unique(), sns.color_palette('tab10')))
fig, ax = plt.subplots(figsize=(9, 0.45 * len(summary) + 1))
sns.barplot(data=summary, y='Modelo', x='Test accuracy', hue='Familia', palette=palette, dodge=False, ax=ax)
ax.set_xlim(0, 1)
ax.set_title('Accuracy en test (mismas 200 canciones)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

**Respuesta Q1:** _(completar tras ejecutar)_

## Q2. ¿Las CNNs y los modelos de descriptores cometen los mismos errores?
Se comparan tres modelos: la mejor CNN desde cero, el mejor fine-tuning (elegido por validación) y el mejor modelo de descriptores (elegido por CV).
- **Matrices de confusión**.
- **Solapamiento de errores**: si es bajo, los enfoques son complementarios.
- **Pares de géneros más confundidos** en cada modelo: si coinciden, la estructura de similitud entre géneros del Lab 1 es propia de la música y no un efecto de los descriptores.

In [ ]:
best_ft_name = max(ft_runs, key=lambda n: ft_runs[n]['val_acc'])
shown = ['CNN desde cero', best_ft_name, best_lab1_name]
y_test = y[idx_test]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, name in zip(axes, shown):
    cm = confusion_matrix(y_test, test_preds[name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=genre_names, yticklabels=genre_names)
    ax.set_title(f'{name}\naccuracy = {accuracy_score(y_test, test_preds[name]):.3f}')
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

In [ ]:
errors = {name: test_preds[name] != y_test for name in shown}

overlap_rows = []
for i, a in enumerate(shown):
    for b in shown[i + 1:]:
        both = (errors[a] & errors[b]).sum()
        either = (errors[a] | errors[b]).sum()
        overlap_rows.append({'Modelo A': a, 'Modelo B': b, 'Errores A': errors[a].sum(), 'Errores B': errors[b].sum(),
                             'Fallan ambos': both, 'Solapamiento (Jaccard)': both / either,
                             'Acierta al menos uno (oráculo)': 1 - both / len(y_test)})
display(pd.DataFrame(overlap_rows).style.format({'Solapamiento (Jaccard)': '{:.2f}', 'Acierta al menos uno (oráculo)': '{:.3f}'}))

def top_confusions(y_true, y_pred, k=5):
    cm = confusion_matrix(y_true, y_pred)
    sym = cm + cm.T  # confusiones en ambos sentidos
    pairs = [(sym[i, j], f'{genre_names[i]} ↔ {genre_names[j]}') for i in range(len(genre_names)) for j in range(i + 1, len(genre_names))]
    return [f'{pair} ({n})' for n, pair in sorted(pairs, reverse=True)[:k]]

print('Pares de géneros más confundidos (nº de canciones):')
display(pd.DataFrame({name: top_confusions(y_test, test_preds[name]) for name in shown}, index=range(1, 6)))

**Respuesta Q2:** _(completar tras ejecutar)_

## Q5. ¿Cuánto dice una sola partición 80/20?
Con 200 canciones de test, una canción vale 0.5 puntos, así que la diferencia entre dos modelos en una sola partición puede deberse al azar. Se repite la evaluación con **5 particiones distintas** (semillas 0-4), manteniendo fija la configuración de cada modelo:

- **CNN desde cero**: la mejor configuración de la Task 1.
- **ResNet18 congelado + LogReg**: los embeddings ya están calculados; solo se reentrena la regresión logística.
- **ResNet18 fine-tuning**.
- **Mejor modelo de descriptores** del Lab 1, con su rejilla de hiperparámetros.

VGG16 fine-tuning se deja fuera de este experimento por coste: es el modelo más lento.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
n_conv_best, filters_best = int(best_row['Capas conv']), int(best_row['Filtros iniciales'])
lab1_model, lab1_params = lab1_models[best_lab1['name']]

seed_rows = []
for seed in SEEDS:
    tr_all, tr, va, te = make_split(seed)
    start = time.time()

    set_seed(seed)
    model, _, _, _ = train_model(SimpleCNN(n_conv_best, filters_best), prep_small,
                                 make_loader(X_small, tr, shuffle=True), make_loader(X_small, va))
    acc_cnn = accuracy_score(*predict(model, prep_small, make_loader(X_small, te)))

    model, _, _, _ = finetune('ResNet18', tr, va, seed=seed)
    acc_ft = accuracy_score(*predict(model, prep_imagenet, make_loader(X_224, te, batch_size=64)))
    del model
    torch.cuda.empty_cache()

    _, _, acc_emb = grid_eval(LogisticRegression(max_iter=2000), LOGREG_GRID, embeddings['ResNet18'], tr_all, te)
    _, _, acc_lab1 = grid_eval(lab1_model, lab1_params, X_desc, tr_all, te)

    for name, acc in [('CNN desde cero', acc_cnn), ('ResNet18 fine-tuning', acc_ft),
                      ('ResNet18 congelado + LogReg', acc_emb), (best_lab1_name, acc_lab1)]:
        seed_rows.append({'Semilla': seed, 'Modelo': name, 'Test accuracy': acc})
    print(f'Semilla {seed}: CNN {acc_cnn:.3f} | FT {acc_ft:.3f} | emb {acc_emb:.3f} | Lab1 {acc_lab1:.3f} '
          f'({time.time() - start:.0f}s)')

seed_df = pd.DataFrame(seed_rows)

In [ ]:
stats = (seed_df.groupby('Modelo')['Test accuracy']
         .agg(['mean', 'std', 'min', 'max']).sort_values('mean', ascending=False))
stats['rango (canciones)'] = ((stats['max'] - stats['min']) * len(idx_test)).round().astype(int)
display(stats.style.format({c: '{:.3f}' for c in ['mean', 'std', 'min', 'max']}))

# Diferencias pareadas: cada modelo contra el mejor de descriptores en la misma partición
pivot = seed_df.pivot(index='Semilla', columns='Modelo', values='Test accuracy')
paired = pd.DataFrame({
    'Diferencia media vs Lab 1': pivot.drop(columns=best_lab1_name).sub(pivot[best_lab1_name], axis=0).mean(),
    'Std de la diferencia': pivot.drop(columns=best_lab1_name).sub(pivot[best_lab1_name], axis=0).std(),
    f'Gana en (de {len(SEEDS)} particiones)': (pivot.drop(columns=best_lab1_name).gt(pivot[best_lab1_name], axis=0)).sum(),
})
display(paired.style.format({'Diferencia media vs Lab 1': '{:+.3f}', 'Std de la diferencia': '{:.3f}'}))

order = stats.index.tolist()
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.stripplot(data=seed_df, x='Modelo', y='Test accuracy', order=order, hue='Semilla', palette='tab10', size=8, jitter=False, ax=ax)
sns.pointplot(data=seed_df, x='Modelo', y='Test accuracy', order=order, color='black', markers='_', errorbar='sd', linestyles='none', ax=ax)
ax.set_title(f'Accuracy en test con {len(SEEDS)} particiones 80/20 distintas (media ± std)')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

**Respuesta Q5:** _(completar tras ejecutar)_